# Independent methodological stress tests

## tl;dr

These tests audit meanings and identification, not conformance with the package specification. On deliberately small known-truth cases:

- Shapley preserves raw signed game values such as `40.5` and `-39.5` while withholding normalized attribution when those values cannot form shares;
- fixed-population Markov `fail` produces bounded shares, while `fail_refit` and `detour` preserve signed diagnostics and report near-cancellation instead of explosive shares;
- identical experimental evidence supports several incompatible calibrated allocations because residual policies select them;
- experimental standard error is stored but does not affect any calibrated value;
- temporal interpolation produces a precise middle value that is not identified by endpoint experiments.

## Context & Methods

The reader is a practitioner deciding whether package outputs can support consequential allocation decisions. We inspect actual package outputs, compare them with known causal truths and transparent baselines, and construct observationally indistinguishable worlds.

### Key Assumptions

- A randomized lift is causal only for its stated population, outcome, intervention, and period.
- Journey attribution is treated as observational unless a separate identification design justifies a causal claim.
- A value assigned by a residual, interpolation, pooling, or weighting policy is a policy-conditioned scenario unless the inputs identify that rule.
- Squared-error comparisons below are illustrative known-truth checks, not claims that squared error is the only decision loss.

In [1]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from do_attribution import MarkovAttribution, ShapleyAttribution
from do_attribution.calibration import (
    AttributionCalibration,
    LiftTestEvidence,
    TemporalAttributionCalibration,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

## Results

### 1. The Shapley output is not a stable allocation of observed conversions

The package defines `v(S)` as conversions on paths whose channel set is **exactly** `S`. Shapley efficiency therefore allocates `v(N)`, the conversions on paths containing every observed channel, rather than all observed conversions. The raw signed game values remain available, but normalization is explicitly unavailable when cancellation or a negative candidate prevents a finite, nonnegative, bounded unit-sum vector.

In [2]:
shapley_rows = []
shapley_cases = {
    "no grand-coalition path": (["A"] * 90 + ["B"] * 10, [1] * 100),
    "add one A>B conversion": (["A"] * 90 + ["B"] * 10 + ["A > B"], [1] * 101),
}
for case_name, (paths, outcomes) in shapley_cases.items():
    with warnings.catch_warnings(record=True) as caught:
        result = ShapleyAttribution().fit(paths, outcomes)
    shapley_rows.append({
        "case": case_name,
        "observed_conversions": sum(outcomes),
        "v_grand_coalition": result["value_function"].get(("A", "B"), 0),
        "raw_A": result["shapley_values"]["A"],
        "raw_B": result["shapley_values"]["B"],
        "share_A": (result["attribution"] or {}).get("A", np.nan),
        "share_B": (result["attribution"] or {}).get("B", np.nan),
        "normalization_status": result["normalization_status"],
        "normalization_reason": result["normalization_reason"],
        "warning": " | ".join(str(item.message) for item in caught),
    })
shapley_results = pd.DataFrame(shapley_rows)
display(shapley_results)

assert shapley_results["share_A"].isna().all()
assert shapley_results["share_B"].isna().all()
assert shapley_results["normalization_reason"].tolist() == [
    "near_cancellation", "negative_share"
]
assert shapley_results[["raw_A", "raw_B"]].to_numpy().tolist() == [
    [40.0, -40.0], [40.5, -39.5]
]

,case,observed_conversions,v_grand_coalition,raw_A,raw_B,share_A,share_B,normalization_status,normalization_reason,warning
0,no grand-coalition path,100,0,40,-40,NaN,NaN,unavailable,near_cancellation,Shapley normalization is unavailable (near_can...
1,add one A>B conversion,101,1,40.5,-39.5,NaN,NaN,unavailable,negative_share,Shapley normalization is unavailable (negative...


### 2. Markov removal policies answer strategy-specific population questions

The observed data below contain 500 A-only journeys with an 80% conversion rate and 500 B-only journeys with a 20% rate. The same records are compatible with at least two causal worlds: randomized A-versus-B assignment with a 60-point treatment contrast, or perfect risk-based selection with zero treatment effect. Any journey-only estimator must return the same answer in both worlds.

In [3]:
markov_paths = ["A"] * 500 + ["B"] * 500
markov_outcomes = [1] * 400 + [0] * 100 + [1] * 100 + [0] * 400
markov_rows = []
for strategy in ("fail", "fail_refit", "detour", "redirect_to_null"):
    with warnings.catch_warnings(record=True) as caught:
        result = MarkovAttribution(removal_strategy=strategy).fit(
            markov_paths, markov_outcomes
        )
    markov_rows.append({
        "strategy": strategy,
        "base_probability": result["total_conversion_probability"],
        "effect_A": result["removal_effects"]["A"],
        "effect_B": result["removal_effects"]["B"],
        "share_A": (result["attribution"] or {}).get("A", np.nan),
        "share_B": (result["attribution"] or {}).get("B", np.nan),
        "normalization_status": result.get("normalization_status", "available"),
        "normalization_reason": result.get("normalization_reason"),
        "population_policy": result.get("population_policy", "fixed"),
        "warning": " | ".join(str(item.message) for item in caught),
    })
markov_results = pd.DataFrame(markov_rows)
display(markov_results)

identical_input_worlds = pd.DataFrame([
    {
        "world": "randomized assignment",
        "observed P(Y=1|A)": 0.8,
        "observed P(Y=1|B)": 0.2,
        "causal A-vs-B effect": 0.6,
    },
    {
        "world": "risk selection; channel has no effect",
        "observed P(Y=1|A)": 0.8,
        "observed P(Y=1|B)": 0.2,
        "causal A-vs-B effect": 0.0,
    },
])
display(identical_input_worlds)

fixed_fail = markov_results.loc[markov_results["strategy"].eq("fail")].iloc[0]
assert fixed_fail["effect_B"] > 0
assert fixed_fail["population_policy"] == "fixed"
for strategy in ("fail_refit", "detour"):
    unsafe = markov_results.loc[markov_results["strategy"].eq(strategy)].iloc[0]
    assert unsafe["normalization_status"] == "unavailable"
    assert unsafe["normalization_reason"] == "near_cancellation"
    assert np.isnan(unsafe["share_A"]) and np.isnan(unsafe["share_B"])
assert markov_results.loc[
    markov_results["strategy"].eq("fail_refit"), "population_policy"
].iloc[0] == "population_refit"

,strategy,base_probability,effect_A,effect_B,share_A,share_B,normalization_status,normalization_reason,population_policy,warning
0,fail,0.5,0.8,0.2,0.8,0.2,available,NaN,fixed,removal_strategy='fail' now uses fixed-populat...
1,fail_refit,0.5,0.6,-0.6,NaN,NaN,unavailable,near_cancellation,population_refit,
2,detour,0.5,-0.6,0.6,NaN,NaN,unavailable,near_cancellation,fixed,
3,redirect_to_null,0.5,0.8,0.2,0.8,0.2,available,NaN,fixed,


,world,observed P(Y=1|A),observed P(Y=1|B),causal A-vs-B effect
0,randomized assignment,0.8,0.2,0.6
1,risk selection; channel has no effect,0.8,0.2,0


### 3. Static calibration identifies one matched tested value, then allocates the rest

Here the matched estimate is A=400 and the valid experimental lift is A=360, so the tested value equals the experiment by construction. The same evidence and attribution inputs yield incompatible B/C values under different residual policies.

In [4]:
original_estimates = {"A": 400.0, "B": 350.0, "C": 250.0}
static_evidence = LiftTestEvidence(
    tested_entity="A",
    observed_lift=360.0,
    matched_mta_estimate=400.0,
    test_id="A-randomized-test",
    outcome_name="incremental conversions",
    population="matched population",
    unit="conversion count",
)
static_calibrator = AttributionCalibration([static_evidence])
policy_calls = {
    "leave untested unchanged": {"residual_policy": "allow_total_change"},
    "proportional untested": {
        "residual_policy": "proportional_untested",
        "eligible_entities": ["B", "C"],
    },
    "all residual to B": {
        "residual_policy": "custom_weights",
        "residual_weights": {"B": 1.0, "C": 0.0},
    },
    "all residual to C": {
        "residual_policy": "custom_weights",
        "residual_weights": {"B": 0.0, "C": 1.0},
    },
    "unidentified bucket": {
        "residual_policy": "residual_bucket",
        "residual_entity": "Unidentified residual",
    },
}
policy_rows = []
for policy_name, arguments in policy_calls.items():
    result = static_calibrator.apply_estimates(original_estimates, **arguments)
    values = {row.entity: row.calibrated_estimate for row in result.rows}
    policy_rows.append({
        "policy": policy_name,
        "A": values.get("A", 0.0),
        "B": values.get("B", 0.0),
        "C": values.get("C", 0.0),
        "unidentified": values.get("Unidentified residual", 0.0),
        "total": result.calibrated_total,
    })
policy_results = pd.DataFrame(policy_rows)
display(policy_results)

assert set(policy_results["A"]) == {360.0}
assert policy_results["B"].nunique() > 1
assert policy_results["C"].nunique() > 1

,policy,A,B,C,unidentified,total
0,leave untested unchanged,360,350,250,0,960
1,proportional untested,360,373.333,266.667,0,"1,000"
2,all residual to B,360,390,250,0,"1,000"
3,all residual to C,360,350,290,0,"1,000"
4,unidentified bucket,360,350,250,40,"1,000"


### 4. No allocation rule dominates transparent baselines

Each truth world below is compatible with the same observed attribution and the same A experiment. Different rules win in different worlds, so experimental evidence on A does not choose among them. Reporting attribution and experiment separately avoids a false point estimate for B and C.

In [5]:
candidate_allocations = {
    "replace tested only": np.array([360.0, 350.0, 250.0]),
    "closed proportional": np.array([360.0, 373.3333333333, 266.6666666667]),
    "rescale all by 0.9": np.array([360.0, 315.0, 225.0]),
    "closed residual to B": np.array([360.0, 390.0, 250.0]),
}
truth_worlds = {
    "untested truly unchanged": np.array([360.0, 350.0, 250.0]),
    "truth matches proportional rule": np.array([360.0, 373.3333333333, 266.6666666667]),
    "truth scales every channel": np.array([360.0, 315.0, 225.0]),
    "truth puts residual on B": np.array([360.0, 390.0, 250.0]),
    "attribution total is not causal total": np.array([360.0, 100.0, 40.0]),
}
error_rows = []
for world_name, truth in truth_worlds.items():
    row = {"truth_world": world_name, "true_total": truth.sum()}
    for method_name, prediction in candidate_allocations.items():
        row[method_name] = float(np.square(prediction - truth).sum())
    error_rows.append(row)
baseline_errors = pd.DataFrame(error_rows)
display(baseline_errors)

winner_columns = list(candidate_allocations)
baseline_errors["lowest_error_rule"] = baseline_errors[winner_columns].idxmin(axis=1)
display(baseline_errors[["truth_world", "true_total", "lowest_error_rule"]])

,truth_world,true_total,replace tested only,closed proportional,rescale all by 0.9,closed residual to B
0,untested truly unchanged,960,0,822.222,"1,850","1,600"
1,truth matches proportional rule,"1,000",822.222,0,"5,138.89",555.556
2,truth scales every channel,900,"1,850","5,138.89",0,"6,250"
3,truth puts residual on B,"1,000","1,600",555.556,"6,250",0
4,attribution total is not causal total,500,"106,600","126,089","80,450","128,200"


,truth_world,true_total,lowest_error_rule
0,untested truly unchanged,960,replace tested only
1,truth matches proportional rule,"1,000",closed proportional
2,truth scales every channel,900,rescale all by 0.9
3,truth puts residual on B,"1,000",closed residual to B
4,attribution total is not causal total,500,rescale all by 0.9


### 5. Uncertainty is recorded but not propagated; transport is assumed

Changing the experimental standard error by a factor of 1,000 leaves every output unchanged. Applying the reference ratio `360/400 = 0.9` to a new A estimate of 800 yields 720; that equals a current causal effect only under an untested multiplicative-transport assumption.

In [6]:
uncertainty_rows = []
for standard_error in (1.0, 1_000.0):
    evidence = LiftTestEvidence(
        tested_entity="A",
        observed_lift=360.0,
        matched_mta_estimate=400.0,
        test_id=f"A-se-{standard_error:g}",
        standard_error=standard_error,
    )
    result = AttributionCalibration([evidence]).apply_estimates(
        original_estimates,
        residual_policy="allow_total_change",
    )
    values = {row.entity: row.calibrated_estimate for row in result.rows}
    uncertainty_rows.append({
        "standard_error": standard_error,
        "calibrated_A": values["A"],
        "calibrated_total": result.calibrated_total,
    })

transport_result = static_calibrator.apply_estimates(
    {"A": 800.0, "B": 350.0, "C": 250.0},
    residual_policy="allow_total_change",
)
transport_A = {row.entity: row.calibrated_estimate for row in transport_result.rows}["A"]
display(pd.DataFrame(uncertainty_rows))
display(pd.DataFrame([
    {"reference_lift": 360.0, "reference_MTA": 400.0, "current_MTA": 800.0, "calibrated_A": transport_A},
]))

assert len({row["calibrated_A"] for row in uncertainty_rows}) == 1
assert transport_A == 720.0

,standard_error,calibrated_A,calibrated_total
0,1,360,960
1,"1,000",360,960


,reference_lift,reference_MTA,current_MTA,calibrated_A
0,360,400,800,720


### 6. Conflict resolution and temporal interpolation add policy, not identification

Two conflicting experiments in one epoch produce 0.5, 1.0, or 1.1667 depending on selection, user weights, or pooled numerator/denominator arithmetic. Separately, endpoint ratios 0.5 and 1.5 produce an interpolated middle ratio of exactly 1.0. A constant, step, delayed, or nonlinear response path is equally compatible with the endpoint evidence.

In [7]:
conflict_evidence = [
    LiftTestEvidence(
        tested_entity="A", observed_lift=50.0, matched_mta_estimate=100.0,
        test_id="low", outcome_name="conversions", population="P", unit="count",
        coordinate_name="x", coordinate_value=5.0,
    ),
    LiftTestEvidence(
        tested_entity="A", observed_lift=300.0, matched_mta_estimate=200.0,
        test_id="high", outcome_name="conversions", population="P", unit="count",
        coordinate_name="x", coordinate_value=5.0,
    ),
]
conflict_calibrator = TemporalAttributionCalibration(conflict_evidence)
conflict_base = dict(
    estimates_by_epoch={"E1": {"A": 100.0, "B": 100.0}},
    ratio_policy="observed_only",
    epoch_ratio_policy="direct",
    residual_policy="allow_total_change",
    epoch_windows={"E1": (0.0, 10.0)},
)
conflict_calls = {
    "select low experiment": {
        "evidence_conflict_policy": "select_evidence",
        "selected_evidence": {("A", "E1"): "low"},
    },
    "equal-weight ratios": {
        "evidence_conflict_policy": "weighted_ratio",
        "evidence_weights": {("A", "E1"): {"low": 0.5, "high": 0.5}},
    },
    "pool lifts and denominators": {"evidence_conflict_policy": "pooled_lift"},
}
conflict_rows = []
for name, arguments in conflict_calls.items():
    result = conflict_calibrator.apply_estimates(**conflict_base, **arguments)
    ratio_row = result.epoch_ratio_schedule[0]
    conflict_rows.append({
        "conflict_policy": name,
        "effective_ratio": ratio_row.effective_ratio,
        "calibrated_A": {row.entity: row.calibrated_estimate for row in result.rows["E1"]}["A"],
    })
display(pd.DataFrame(conflict_rows))

temporal_evidence = [
    LiftTestEvidence(
        tested_entity="A", observed_lift=50.0, matched_mta_estimate=100.0,
        test_id="left", coordinate_name="x", coordinate_value=0.0,
    ),
    LiftTestEvidence(
        tested_entity="A", observed_lift=150.0, matched_mta_estimate=100.0,
        test_id="right", coordinate_name="x", coordinate_value=10.0,
    ),
]
temporal_result = TemporalAttributionCalibration(temporal_evidence).apply_estimates(
    {
        "left epoch": {"A": 100.0, "B": 100.0},
        "middle epoch": {"A": 100.0, "B": 100.0},
        "outside epoch": {"A": 100.0, "B": 100.0},
    },
    ratio_policy="coordinate_linear",
    epoch_ratio_policy="representative_coordinate",
    residual_policy="allow_total_change",
    epoch_windows={
        "left epoch": (0.0, 2.0),
        "middle epoch": (4.0, 6.0),
        "outside epoch": (12.0, 14.0),
    },
    epoch_coordinates={"left epoch": 0.0, "middle epoch": 5.0, "outside epoch": 13.0},
)
temporal_rows = [
    {
        "epoch": epoch,
        "effective_ratio": next((row.effective_ratio for row in temporal_result.epoch_ratio_schedule if row.epoch == epoch), np.nan),
        "ratio_source": next((row.ratio_source for row in temporal_result.epoch_ratio_schedule if row.epoch == epoch), "not calibrated"),
        "calibrated_A": {row.entity: row.calibrated_estimate for row in temporal_result.rows[epoch]}["A"],
    }
    for epoch in temporal_result.epochs
]
display(pd.DataFrame(temporal_rows))

middle_ratio = next(row.effective_ratio for row in temporal_result.epoch_ratio_schedule if row.epoch == "middle epoch")
assert middle_ratio == 1.0
assert all(row.epoch != "outside epoch" for row in temporal_result.epoch_ratio_schedule)

,conflict_policy,effective_ratio,calibrated_A
0,select low experiment,0.5,50
1,equal-weight ratios,1,100
2,pool lifts and denominators,1.16667,116.667


,epoch,effective_ratio,ratio_source,calibrated_A
0,left epoch,0.5,observed,50
1,middle epoch,1,interpolated,100
2,outside epoch,NaN,not calibrated,100


## Takeaways

1. **Journey attribution is descriptive and non-identified for causality.** The same observed paths and outcomes can arise under different causal effects.
2. **Unsafe Shapley and Markov normalization is unavailable, not fabricated.** Raw signed diagnostics and machine-readable reasons remain visible; fixed-population `fail` is distinct from population-changing `fail_refit`.
3. **Static calibration has one narrow valid case:** when the current tested estimate exactly matches the experiment denominator, direct calibration replaces that tested value with the experimental point estimate. The package does not propagate its uncertainty.
4. **Untested changes are allocations.** The evidence does not select the recipient set, weights, preserved total, or residual destination.
5. **Temporal ratios are scenarios between anchors.** Smoothness and deterministic provenance do not identify the response path.
6. **Simple separate reporting is often more informative.** It exposes what is known without completing the table with unsupported point values.